In [23]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

# φόρτωση δεδομένων σε dataframe
data9 = pd.read_csv('S024.csv')
#data23 = pd.read_csv('S023.csv')
#data15 = pd.read_csv('S015.csv')
#data10 = pd.read_csv('S010.csv')

# Κανονικοποίηση δεδομένων
data9[['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z']] = (
    data9[['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z']]
    .apply(lambda x: (x - x.mean()) / x.std())
)

sampling_frequency = 8 #η συχνοτητα δειγματοληψιας - μετρησεις ανα δευτερόλεπτο
time_interval = 2  #σβηνουμε μια γραμμη καθε 2 δευτερολεπτα για να μειωσουμε τον χρονο εκπαιδευσης των μοντελων

#Διαδικασια αφαιρεσης γραμμων
rows_to_remove9 = range( 8, time_interval // sampling_frequency)
#rows_to_remove23 = range(0, len(data23), time_interval // sampling_frequency)
#rows_to_remove15 = range(0, len(data15), time_interval // sampling_frequency)
#rows_to_remove10 = range(0, len(data10), time_interval // sampling_frequency)

# καινουργιο datframe με το μειωμενο αρχεο.
#data_reduced10 = data10.drop(rows_to_remove10).reset_index(drop=True)
data_reduced9 = data9.drop(rows_to_remove9).reset_index(drop=True)
#data_reduced23 = data23.drop(rows_to_remove23).reset_index(drop=True)
#data_reduced15 = data15.drop(rows_to_remove15).reset_index(drop=True)


# Εδω εγινε μια προσπαθεία να συχνονευτουν τα αρχεια και να τρεξουν ολα μαζι αλλα δεν ειχα τον απαιτουμενο χωρο 
#merged_data = data_reduced10.merge(data_reduced9, on='label', suffixes=('10', '9'))
#merged_data2 = merged_data.merge(data_reduced23, on='label', suffixes=('', '23'))
#merged_data1 = merged_data2.merge(data_reduced15, on='label', suffixes=('', '15'))


# Χρειαζομαστε τια παρακάτω στηλες για χρήση στο sliding window
features = ['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z']
label_column = 'label'  

#ελεγχος για  σωστες στηλες στο Merge αρχειο
#if not set(features).issubset(rows_to_remove9.columns):
    #raise ValueError("Some of the required features are missing in the merged data.")

#if label_column not in rows_to_remove9.columns:
    #raise ValueError(f"The column '{label_column}' is missing from the merged data.")

# create sliding window
window_size = 10

def create_sliding_window(data, window_size): #χωρίζει το dataframe σε τμήματα
    for i in range(len(data) - window_size):
        X.append(data[features].iloc[i:i + window_size].values.flatten())
        y.append(data[label_column].iloc[i + window_size - 1])
    return pd.DataFrame(X), pd.Series(y)

X, y = create_sliding_window(data_reduced9, window_size)

# δημιουργούμε training set = 70% και test set = 30% για την εκπαίδευση και τον ελεγχο των μοντέλων
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Εκπαίδευση Random Forest
rf_model = RandomForestClassifier(n_estimators=100) 
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)


print("Random Forest Classifier:")
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

# Εκπαίδευση Neural Network
nn_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300)
nn_model.fit(X_train, y_train)
nn_pred = nn_model.predict(X_test)


print("\nNeural Network Classifier:")
print(confusion_matrix(y_test, nn_pred))
print(classification_report(y_test, nn_pred))

# Εκπαίδευση Bayesian Classifier
bayes_model = GaussianNB()
bayes_model.fit(X_train, y_train)
bayes_pred = bayes_model.predict(X_test)


print("\nBayesian Classifier:")
print(confusion_matrix(y_test, bayes_pred))
print(classification_report(y_test, bayes_pred))


Random Forest Classifier:
[[23526    76    12    12    28    66     0     0]
 [  142  7942     0     0     1     0     0     0]
 [  446     6   430     0     0   137     0     0]
 [  236     6     0   762     2     0     0     0]
 [  180     4     0     0   682     0     0     0]
 [  338    14    35     0     0  2736     0     0]
 [    2     0     0     0     3     0  4161     1]
 [    0     0     0     0     0     0     1  9171]]
              precision    recall  f1-score   support

           1       0.95      0.99      0.97     23720
           2       0.99      0.98      0.98      8085
           3       0.90      0.42      0.57      1019
           4       0.98      0.76      0.86      1006
           5       0.95      0.79      0.86       866
           6       0.93      0.88      0.90      3123
           7       1.00      1.00      1.00      4167
           8       1.00      1.00      1.00      9172

    accuracy                           0.97     51158
   macro avg       0.96

C:\Users\maria\PycharmProjects\pythonProject\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(



Neural Network Classifier:
[[23050   109   241    81    74   163     2     0]
 [  172  7877     4    13    12     5     2     0]
 [  333     2   466     6     5   207     0     0]
 [  162    15    12   811     3     3     0     0]
 [  169    15     6     9   667     0     0     0]
 [  218     6   195     4     0  2700     0     0]
 [    1     0     0     0     0     0  4166     0]
 [    0     0     0     0     0     0     0  9172]]
              precision    recall  f1-score   support

           1       0.96      0.97      0.96     23720
           2       0.98      0.97      0.98      8085
           3       0.50      0.46      0.48      1019
           4       0.88      0.81      0.84      1006
           5       0.88      0.77      0.82       866
           6       0.88      0.86      0.87      3123
           7       1.00      1.00      1.00      4167
           8       1.00      1.00      1.00      9172

    accuracy                           0.96     51158
   macro avg       0.